In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, count, countDistinct, when, isnan, isnull
# from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import Imputer
# from pyspark.ml.regression import LinearRegression
import pyspark.sql.functions as F
import boto3
import pandas as pd

In [2]:
# START THE SPARK SESSION WITH HUDI
# Changes will be needed so it can interact with GLUE
# SPARK VERSION 3.4.2 IS REQUIRED ON THE SYSTEM TO INTERACT WITH HUDI

spark = SparkSession.builder \
    .appName("Hudi-app") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog") \
    .config("spark.sql.hive.convertMetastoreParquet", "false") \
    .config('spark.kryo.registrator', 'org.apache.spark.HoodieSparkKryoRegistrar') \
    .config('spark.sql.extensions', 'org.apache.spark.sql.hudi.HoodieSparkSessionExtension') \
    .config(
        'spark.jars.packages',
        'org.apache.hudi:hudi-spark3-bundle_2.12:0.14.1'
    ) \
    .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
24/03/11 14:01:15 WARN Utils: Your hostname, 21310-466 resolves to a loopback address: 127.0.1.1; using 172.26.135.218 instead (on interface eth0)
24/03/11 14:01:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/sgomez/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/sgomez/.ivy2/cache
The jars for the packages stored in: /home/sgomez/.ivy2/jars
org.apache.hudi#hudi-spark3-bundle_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9757aeb7-b645-4147-b6a7-548e1817b1df;1.0
	confs: [default]
	found org.apache.hudi#hudi-spark3-bundle_2.12;0.14.1 in central
:: resolution report :: resolve 131ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.hudi#hudi-spark3-bundle_2.12;0.14.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-9757

In [3]:
# LOAD DYNAMODB DATA INTO A SPARK DATAFRAME

# Read DynamoDB
dynamodb = boto3.client('dynamodb')
dynamodb_table_name = 'TelemetryDataTable'

# Scan DynamoDB and extract data from the response
response = dynamodb.scan(
    TableName=dynamodb_table_name,
    Limit=10000)
items = response['Items']

# Convert data so Spark can parse it
data = []
for item in items:
    data.append({key: list(value.values())[0] for key, value in item.items()})

# Create a Spark DataFrame from the data
df = spark.createDataFrame(data)

# Cast values to numeric types where needed
df = df.withColumn("temperature", col("temperature").cast("float")) \
       .withColumn("humidity", col("humidity").cast("float")) \
       .withColumn("pressure", col("pressure").cast("float"))

In [12]:
df.printSchema()
df.toPandas().head()

root
 |-- device_id: string (nullable = true)
 |-- humidity: float (nullable = true)
 |-- light: boolean (nullable = true)
 |-- pressure: float (nullable = true)
 |-- temperature: float (nullable = true)
 |-- timestamp: string (nullable = true)



,device_id,humidity,light,pressure,temperature,timestamp
0,device_2,60.009998,False,1099.400024,11.140000,2024-03-11T09:36:58.027937
1,device_2,NaN,True,1023.830017,36.810001,2024-03-11T09:36:59.029302
2,device_2,30.309999,True,926.969971,NaN,2024-03-11T09:37:01.031100
3,device_2,74.879997,False,1040.579956,34.380001,2024-03-11T09:37:03.034595
4,device_2,33.009998,True,953.679993,13.440000,2024-03-11T09:37:04.036204


In [4]:
# GLOBAL VARIABLES

COW_TABLE_NAME = 'telemetry_table'
BASE_PATH = 'file:////home/sgomez/proyectos/2023-sergio_gomez/tests/hudi-table'
# Later this will be the S3 path where Hudi tables are stored
PARTITION_FIELD = 'device_id'
RECORD_KEY = ['device_id', 'timestamp']

In [5]:
# INITIALIZE HUDI
# CHECK THAT ALL OPTIONS MATCH THE USE CASE

# Options for Hudi tables
hudi_options = {
    'hoodie.table.name': COW_TABLE_NAME,
    'hoodie.datasource.write.storage.type': 'COPY_ON_WRITE',
    'hoodie.datasource.write.partitionpath.field': PARTITION_FIELD,
    'hoodie.datasource.write.recordkey.field': ','.join(RECORD_KEY)
}

# Initial write to the Hudi table
df.write.format('hudi') \
    .option('hoodie.datasource.write.operation', 'insert') \
    .options(**hudi_options) \
    .mode('overwrite') \
    .save(BASE_PATH)

24/03/11 14:02:17 WARN DFSPropertiesConfiguration: Cannot find HUDI_CONF_DIR, please set it as the dir of hudi-defaults.conf
24/03/11 14:02:17 WARN DFSPropertiesConfiguration: Properties file file:/etc/hudi/conf/hudi-defaults.conf not found. Ignoring to load props file
24/03/11 14:02:17 WARN HoodieSparkSqlWriterInternal: hoodie table at file:/home/sgomez/proyectos/2023-sergio_gomez/tests/hudi-table already exists. Deleting existing data & overwriting with new data.
24/03/11 14:02:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-hbase.properties,hadoop-metrics2.properties


# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf


# WARNING: Unable to attach Serviceability Agent. Unable to attach even with module exceptions: [org.apache.hudi.org.openjdk.jol.vm.sa.SASupportException: Sense failed., org.apache.hudi.org.openjdk.jol.vm.sa.SASupportException: Sense failed., org.apache.hudi.org.openjdk.jol.vm.sa.SASupportException: Sense failed.]


24/03/11 14:02:27 WARN HoodieSparkSqlWriterInternal: Closing write client       


In [6]:
# DELETE ENTRIES WHERE DEVICE_ID OR TIMESTAMP IS NULL

nullKeysDF = spark.read.format('hudi').load(BASE_PATH)

filtered_nullKeysDF = nullKeysDF.filter(col('device_id').isNull() | col('timestamp').isNull())

# HUDI DELETE OPTIONS
hudi_delete_options = {
    'hoodie.table.name': COW_TABLE_NAME,
    'hoodie.table.type': 'COPY_ON_WRITE',
    'hoodie.datasource.write.partitionpath.field': PARTITION_FIELD,
    'hoodie.datasource.write.recordkey.field': ','.join(RECORD_KEY),
    'hoodie.datasource.write.operation': 'delete'
}

# Delete rows where device_id or timestamp is null
filtered_nullKeysDF.write.format('hudi') \
    .options(**hudi_delete_options) \
    .mode('append') \
    .save(BASE_PATH)

24/03/11 14:02:42 WARN HoodieSparkSqlWriterInternal: Closing write client


In [7]:
# DELETE DATA
# Delete tuples that have more null values than valid values
# since those records are not useful
# THIS MUST RUN BEFORE NULL IMPUTATION, OTHERWISE IT DOES NOT MAKE SENSE

moreNullsDF = spark.read.format('hudi').load(BASE_PATH)

# Exclude device_id and timestamp from the calculation (counting them is not meaningful)
excluded_columns = ['device_id', 'timestamp']
to_count_columns = [col for col in moreNullsDF.columns if col not in excluded_columns]

# Count the number of columns and set threshold to half
num_col = len(moreNullsDF.columns)
threshold = num_col // 2

# Count how many nulls each tuple has
null_counts = sum(col(c).isNull().cast("int").alias(c) for c in to_count_columns)

# DataFrame with rows that have more nulls than valid values
filtered_moreNullsDF = moreNullsDF.filter(null_counts >= threshold)

# Delete rows found in the DataFrame with excessive null values
filtered_moreNullsDF.write.format('hudi') \
    .options(**hudi_delete_options) \
    .mode('append') \
    .save(BASE_PATH)

24/03/11 14:02:47 WARN HoodieSparkSqlWriterInternal: Closing write client


In [16]:
# TO IMPUTE NULL DATA
df = spark.read.format('hudi').load(BASE_PATH)

# Define numeric columns where mean imputation will be applied
numeric_columns = ['temperature', 'pressure', 'humidity']

# Impute null values in numeric columns using the mean
imputer_mean = Imputer(inputCols=numeric_columns, outputCols=numeric_columns)
imputed_df = imputer_mean.setStrategy("mean").fit(df).transform(df)

imputed_df.write.format('hudi') \
    .option('hoodie.datasource.write.operation', 'insert') \
    .options(**hudi_options) \
    .mode('append') \
    .save(BASE_PATH)

,_hoodie_commit_time,_hoodie_commit_seqno,_hoodie_record_key,_hoodie_partition_path,_hoodie_file_name,humidity,light,pressure,temperature,timestamp,device_id
0,20240311141517272,20240311141517272_0_0,"device_id:device_1,timestamp:2024-03-11T09:36:...",device_1,64cd37f1-2d85-44ff-b54e-ddd20608d6ec-0_0-105-1...,66.201492,False,902.200012,33.009998,2024-03-11T09:36:55.022964,device_1
1,20240311141517272,20240311141517272_0_1,"device_id:device_1,timestamp:2024-03-11T09:36:...",device_1,64cd37f1-2d85-44ff-b54e-ddd20608d6ec-0_0-105-1...,37.040001,True,900.679993,16.150000,2024-03-11T09:36:56.024662,device_1
2,20240311141517272,20240311141517272_0_2,"device_id:device_1,timestamp:2024-03-11T09:37:...",device_1,64cd37f1-2d85-44ff-b54e-ddd20608d6ec-0_0-105-1...,97.320000,True,994.376343,39.070000,2024-03-11T09:37:00.029652,device_1
3,20240311141517272,20240311141517272_0_3,"device_id:device_1,timestamp:2024-03-11T09:37:...",device_1,64cd37f1-2d85-44ff-b54e-ddd20608d6ec-0_0-105-1...,42.459999,False,994.376343,22.549999,2024-03-11T09:37:02.032886,device_1
4,20240311141517272,20240311141517272_0_4,"device_id:device_1,timestamp:2024-03-11T09:37:...",device_1,64cd37f1-2d85-44ff-b54e-ddd20608d6ec-0_0-105-1...,86.589996,None,920.140015,25.283150,2024-03-11T09:37:05.037614,device_1
